# DIP-STER Experiments Juypter Notebook 

## Imports and initialization


In [1]:
import torch
import numpy as np
import dipster_bare as dip
from dipster_bare.data import Sinogram, normalize
import os
from datetime import datetime
import stackview
from pathlib import Path
import mrcfile
import imageio.v2 as io
import json
from dipster_bare.tune import tune as optuna_tune, split_frames
import optuna
import optuna.visualization as vis

%load_ext autoreload
%autoreload 2

In [2]:
torch.set_float32_matmul_precision('high')
torch.cuda.is_available()

True

## Loading tiltseries

In [3]:
# Read the data as tif
# For dipster, the rotation axis should be horizontal as seen in stackview
fname = r"C:\Users\rgirod\Data\20260611_DIPSTER\dipster_bare\data\20260701_NS220_melting\TiltSeries\Melting_0_66_133_200_0.7Migration_GRS_100projs_nodist\Sino_GRS_100projs_nodist.tif"
data = io.volread(fname)

# Load time and angle; this code assumes a 2 columns .csv [angles, times] and will ignore the header line
t = np.loadtxt(r"C:\Users\rgirod\Data\20260611_DIPSTER\dipster_bare\data\20260701_NS220_melting\TiltSeries\Melting_0_66_133_200_0.7Migration_GRS_100projs_nodist\Sino_GRS_100projs_nodist.csv", skiprows=1, delimiter=",")
angles = t[:,0]
times = t[:,1]

print(f'Loaded a {data.shape[1:]} tilt series with {data.shape[0]} projections')
stackview.slice(data)

Loaded a (128, 128) tilt series with 100 projections


In [4]:
# For dipster, the rotation axis should be horizontal as seen in stackview
# If not, run this cell once
# For clarity, the series will be displayed sorted by angle, but the data to be used remain sorted by time
data = np.rot90(data, axes = (1,2))

idx = np.argsort(angles)
stackview.slice(data[idx,...])

In [5]:
# Load the data in a sinogram helper
ts = Sinogram(data = data[:],
              angles = angles[:],
              times = times[:])

### Preprocessing

In [6]:
# Normalize and transfer to GPU
ts = normalize(ts)
print(ts.data.max())

# /!\ Critical point: DIP is in matlab convention whereas stackview is in convention for napari
ts.data = np.transpose(ts.data, (1,2,0))

#convert numpy to tensor
ts.times = np.array(ts.times).astype('float64')
ts.times = (ts.times-np.min(ts.times))/(np.max(ts.times)-np.min(ts.times))
ts.data = torch.from_numpy(ts.data).to("cuda")
ts.angles =torch.from_numpy(ts.angles).to("cuda")
ts.times = torch.from_numpy(ts.times).to("cuda")

print(ts.data.shape)
print(ts.times.shape)
print(ts.angles.shape)

1.0
torch.Size([128, 128, 100])
torch.Size([100])
torch.Size([100])


## Hyperparameter search (Optuna)

Default search includes four hyperparameters of the network:

| param | role | search space |
|-------|------|--------------|
| `style_size` | latent grid size | {4, 8, 16, 32} |
| `depth` | MappingNet FC layers | int [1, 6] |
| `hidden_dim` | MappingNet FC width | {128, 256, 512, 1024} |
| `noise_regularizer` | total-variation weight | log-uniform [1e-7, 1e-2] |

`up_factor` is set as `proj_size // style_size`, so the CNN output stays the same shape as the projections

**Objective:** optimization by held-out validation. Each candidate is
scored by the mean projection-space PSNR on the held-out frames (a full
reconstructed volume is re-projected and compared to the measured projections).
This rewards `noise_regularizer` for generalizing to unseen angles, instead of
driving it to zero.

Note that the search runs without wandb (`hypertrain=True`), so there is no HTML/banner
output. Trials are sequential (one GPU). Adjust `trials` / `n_steps` to your time
budget.

The study is cached in `dipster_tune.db` and is resumable. The progress can be followed in real time with the _optuna dashboard_ extension (right click `.db` file -> open in optuna dashboard)

In [ ]:
# Fixed settings shared by every trial (the params to be tuned are sampled by Optuna)
base_params = dict(
    gamma = 0.8,
    batch_size = 4,
    step_size = 6500,     # StepLR period during the short tuning runs
    output_activation = 'relu',
    compile_net = True
)

study = optuna_tune(
    ts,
    trials = 20,                    # number of candidate configs to try (sequential)
    n_steps = 100000,               # training steps per trial
    save_period = 1000,             # evaluation frequency, will influence the prune cadence
    n_val = 10,                     # held-out validation frames, in %
    base_params = base_params,
    style_sizes=(8,16),             # Any of the optimizable hyperparam can be explicitely declared, or left silent which will use the default range in the table above
    storage = "sqlite:///dipster_tune.db",   # persists / resumes the study
    study_name = "dipster_AuAg_large_2"
)

[I 2026-06-24 23:18:35,978] A new study created in RDB with name: dipster_AuAg_large_2


Frames: 102 total -> 92 train, 10 val [0, 11, 22, 33, 44, 56, 67, 78, 89, 101]
06/24/2026 11:18:36 PM proj_size: 128
06/24/2026 11:18:36 PM frames: 102
06/24/2026 11:18:36 PM dev: cuda:0
06/24/2026 11:18:36 PM seed: 0
06/24/2026 11:18:36 PM opt_over: net
06/24/2026 11:18:36 PM latent_dim: 4
06/24/2026 11:18:36 PM hidden_dim: 512
06/24/2026 11:18:36 PM style_size: 8
06/24/2026 11:18:36 PM depth: 5
06/24/2026 11:18:36 PM Nr: 1
06/24/2026 11:18:36 PM input_nch: 1
06/24/2026 11:18:36 PM output_nch: 1
06/24/2026 11:18:36 PM need_bias: False
06/24/2026 11:18:36 PM up_factor: 16
06/24/2026 11:18:36 PM upsample_mode: nearest
06/24/2026 11:18:36 PM output_activation: relu
06/24/2026 11:18:36 PM lr: 6.260326907499729e-05
06/24/2026 11:18:36 PM step_size: 5000
06/24/2026 11:18:36 PM gamma: 0.75
06/24/2026 11:18:36 PM batch_size: 4
06/24/2026 11:18:36 PM max_steps: 25000
06/24/2026 11:18:36 PM affine_start_step: None
06/24/2026 11:18:36 PM affine_lr: 0.001
06/24/2026 11:18:36 PM affine_regularizer

[I 2026-06-24 23:36:42,795] Trial 0 finished with value: 0.8624859007909655 and parameters: {'lr': 6.260326907499729e-05, 'style_size': 8, 'depth': 5, 'hidden_dim': 512, 'noise_regularizer': 3.75097722181006e-05}. Best is trial 0 with value: 0.8624859007909655.


06/24/2026 11:36:42 PM proj_size: 128
06/24/2026 11:36:42 PM frames: 102
06/24/2026 11:36:42 PM dev: cuda:0
06/24/2026 11:36:42 PM seed: 0
06/24/2026 11:36:42 PM opt_over: net
06/24/2026 11:36:42 PM latent_dim: 4
06/24/2026 11:36:42 PM hidden_dim: 512
06/24/2026 11:36:42 PM style_size: 8
06/24/2026 11:36:42 PM depth: 6
06/24/2026 11:36:42 PM Nr: 1
06/24/2026 11:36:42 PM input_nch: 1
06/24/2026 11:36:42 PM output_nch: 1
06/24/2026 11:36:42 PM need_bias: False
06/24/2026 11:36:42 PM up_factor: 16
06/24/2026 11:36:42 PM upsample_mode: nearest
06/24/2026 11:36:42 PM output_activation: relu
06/24/2026 11:36:42 PM lr: 0.00030374980367128465
06/24/2026 11:36:42 PM step_size: 5000
06/24/2026 11:36:42 PM gamma: 0.75
06/24/2026 11:36:42 PM batch_size: 4
06/24/2026 11:36:42 PM max_steps: 25000
06/24/2026 11:36:42 PM affine_start_step: None
06/24/2026 11:36:42 PM affine_lr: 0.001
06/24/2026 11:36:42 PM affine_regularizer: 1.0
06/24/2026 11:36:42 PM affine_frames_per_step: 1
06/24/2026 11:36:42 PM 

[I 2026-06-24 23:53:16,421] Trial 1 finished with value: 0.8380295868704974 and parameters: {'lr': 0.00030374980367128465, 'style_size': 8, 'depth': 6, 'hidden_dim': 512, 'noise_regularizer': 0.0003549468128702951}. Best is trial 0 with value: 0.8624859007909655.


...

06/25/2026 01:16:28 AM proj_size: 128
06/25/2026 01:16:28 AM frames: 102
06/25/2026 01:16:28 AM dev: cuda:0
06/25/2026 01:16:28 AM seed: 0
06/25/2026 01:16:28 AM opt_over: net
06/25/2026 01:16:28 AM latent_dim: 4
06/25/2026 01:16:28 AM hidden_dim: 512
06/25/2026 01:16:28 AM style_size: 8
06/25/2026 01:16:28 AM depth: 4
06/25/2026 01:16:28 AM Nr: 1
06/25/2026 01:16:28 AM input_nch: 1
06/25/2026 01:16:28 AM output_nch: 1
06/25/2026 01:16:28 AM need_bias: False
06/25/2026 01:16:28 AM up_factor: 16
06/25/2026 01:16:28 AM upsample_mode: nearest
06/25/2026 01:16:28 AM output_activation: relu
06/25/2026 01:16:28 AM lr: 5.738011584290251e-05
06/25/2026 01:16:28 AM step_size: 5000
06/25/2026 01:16:28 AM gamma: 0.75
06/25/2026 01:16:28 AM batch_size: 4
06/25/2026 01:16:28 AM max_steps: 25000
06/25/2026 01:16:28 AM affine_start_step: None
06/25/2026 01:16:28 AM affine_lr: 0.001
06/25/2026 01:16:28 AM affine_regularizer: 1.0
06/25/2026 01:16:28 AM affine_frames_per_step: 1
06/25/2026 01:16:28 AM c

[I 2026-06-25 01:16:56,335] Trial 28 pruned. 


06/25/2026 01:16:56 AM proj_size: 128
06/25/2026 01:16:56 AM frames: 102
06/25/2026 01:16:56 AM dev: cuda:0
06/25/2026 01:16:56 AM seed: 0
06/25/2026 01:16:56 AM opt_over: net
06/25/2026 01:16:56 AM latent_dim: 4
06/25/2026 01:16:56 AM hidden_dim: 512
06/25/2026 01:16:56 AM style_size: 8
06/25/2026 01:16:56 AM depth: 6
06/25/2026 01:16:56 AM Nr: 1
06/25/2026 01:16:56 AM input_nch: 1
06/25/2026 01:16:56 AM output_nch: 1
06/25/2026 01:16:56 AM need_bias: False
06/25/2026 01:16:56 AM up_factor: 16
06/25/2026 01:16:56 AM upsample_mode: nearest
06/25/2026 01:16:56 AM output_activation: relu
06/25/2026 01:16:56 AM lr: 9.410193681640334e-05
06/25/2026 01:16:56 AM step_size: 5000
06/25/2026 01:16:56 AM gamma: 0.75
06/25/2026 01:16:56 AM batch_size: 4
06/25/2026 01:16:56 AM max_steps: 25000
06/25/2026 01:16:56 AM affine_start_step: None
06/25/2026 01:16:56 AM affine_lr: 0.001
06/25/2026 01:16:56 AM affine_regularizer: 1.0
06/25/2026 01:16:56 AM affine_frames_per_step: 1
06/25/2026 01:16:56 AM c

[I 2026-06-25 01:17:10,577] Trial 29 pruned. 



Best value (held-out projection PSNR): 0.8626
Best params: {'lr': 3.3513943895577885e-05, 'style_size': 8, 'depth': 5, 'hidden_dim': 512, 'noise_regularizer': 4.7082625274421616e-05}
  derived up_factor: 16


In [18]:
# Inspect the search results
print("Best held-out projection PSNR:", study.best_value)
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("  derived up_factor:", study.best_trial.user_attrs.get("up_factor"))

# Top configurations
df = study.trials_dataframe()
df.sort_values("value", ascending=False).head(10)

Best held-out projection PSNR: 0.862621545779259
  lr: 3.3513943895577885e-05
  style_size: 8
  depth: 5
  hidden_dim: 512
  noise_regularizer: 4.7082625274421616e-05
  derived up_factor: 16


,number,value,datetime_start,datetime_complete,duration,params_depth,params_hidden_dim,params_lr,params_noise_regularizer,params_style_size,user_attrs_up_factor,state
17,17,0.862622,2026-06-25 00:57:55.660003,2026-06-25 01:09:40.230026,0 days 00:11:44.570023,5,512,0.000034,0.000047,8,16,COMPLETE
0,0,0.862486,2026-06-24 23:18:35.982851,2026-06-24 23:36:42.786249,0 days 00:18:06.803398,5,512,0.000063,0.000038,8,16,COMPLETE
12,12,0.861929,2026-06-25 00:44:58.358615,2026-06-25 00:56:45.467208,0 days 00:11:47.108593,5,512,0.000047,0.000022,8,16,COMPLETE
4,4,0.857230,2026-06-25 00:28:03.930850,2026-06-25 00:39:54.844815,0 days 00:11:50.913965,6,512,0.000055,0.000005,8,16,COMPLETE
3,3,0.850954,2026-06-25 00:11:17.978865,2026-06-25 00:28:03.922343,0 days 00:16:45.943478,3,256,0.000198,0.000388,16,8,COMPLETE
24,24,0.843902,2026-06-25 01:11:32.431121,2026-06-25 01:15:33.258307,0 days 00:04:00.827186,5,512,0.000148,0.000030,8,16,PRUNED
1,1,0.838030,2026-06-24 23:36:42.801589,2026-06-24 23:53:16.415533,0 days 00:16:33.613944,6,512,0.000304,0.000355,8,16,COMPLETE
5,5,0.808531,2026-06-25 00:39:54.853855,2026-06-25 00:42:26.438231,0 days 00:02:31.584376,6,256,0.000086,0.000037,16,8,PRUNED
19,19,0.791029,2026-06-25 01:09:54.206136,2026-06-25 01:10:22.176820,0 days 00:00:27.970684,5,512,0.000032,0.000051,8,16,PRUNED
2,2,0.786180,2026-06-24 23:53:16.425042,2026-06-25 00:11:17.965354,0 days 00:18:01.540312,6,512,0.000007,0.000453,8,16,COMPLETE


In [11]:
vis.plot_optimization_history(study)

In [13]:
vis.plot_slice(study, params=["style_size", "depth", "hidden_dim", "noise_regularizer"])

## Fully train the best model

### In case of crash
In the wandb folder there are auto saved networks every 10000 iterations - you can restart network from there if necesseary

In [ ]:
# net = dip.Solver.from_state_dict(torch.load('/mnt/d/amoncomble/Programs_Data/DIPSTER/Model128/NS2_220_byIndex_rot90_no1st_golden-moon-3_33600.pkl', map_location=torch.device('cpu')))
# net.params.noise_regularizer = 0.000005
# for key, value in net.params.to_dict().items():
#     print(key, value)
# net.eval()
# net.train(ts)

# out_name = ts_name +'_'+net.params.wandb_name
# model_path = os.path.join(directory, model_dir, out_name +'.pkl')
# torch.save(net.state_dict(), model_path)
# newnet = dip.Solver.from_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

# result = new_vmf(os.path.join(directory, out_dir, out_name +'.vmf'))
# for variables in net.reconstruct(ts=ts, slice_by_slice=True):
#     result.write_record(*variables)

### Standard Network Operations

In [ ]:
# The most common parameters are exposed here, the full list and default values are in params.py
# Initialize the solver ------------------------------------------------------------------------------------------------------------
net = dip.Solver(ts)
vol = None

# Apply the best hyperparameters found by Optuna ------------------------------------------------------------------------------------
best = study.best_params

# Architecture params (up_factor derived to keep the 128x128 output)
net.params.style_size = best["style_size"]                          # "width" of the first CNN layer <-> style^2 is the size of the latent vector
net.params.up_factor  = net.params.proj_size // best["style_size"]  # Controls the CNN depth, output size is style_size * up_factor
net.params.depth      = best["depth"]                               # num FC layers
net.params.hidden_dim = best["hidden_dim"]                          # size FC layers
net.params.input_nch  = 1                                           # channels, default is 1

# Priors
net.params.noise_regularizer = best["noise_regularizer"]

In [ ]:
# OR set them manually
# Architecture params
net.params.style_size= 8        # "width" of the first CNN layer <-> style^2 is the size of the latent vector
net.params.input_nch = 1        # Controls the CNN depth, output size is style_size * up_factor
net.params.up_factor = 16       # up from style_size = cnn depth
net.params.depth = 2            # num FC layers
net.params.hidden_dim = 512     # size FC layers

# Priors
net.params.noise_regularizer = 0

In [ ]:
# Other training params
epochs = None
net.params.max_steps = 150000
net.params.batch_size = 4
net.params.gamma = 0.85
net.params.lr = 1e-4
net.params.step_size = int(0.0625 * net.params.max_steps)
net.params.save_period = 1000

# Saving properties
directory = r'C:\Users\rgirod\Data\20260611_DIPSTER\dipster_bare\data\20260615-1245_NS2_TestRuns\Results' # General folder

# Wandb logging
net.params.wandb_local_dir = directory # path to log, will hapen in directory \wandb
net.params.wandb_project = 'Training_my_sample'

# --------------------------------------------------------------------------------------------------------------------------
# Initialize network
print( f'Epochs: {epochs}, Max Steps: {net.params.max_steps}, Step Size: {net.params.step_size}')
meta = net.params.to_dict()
net._setup(ts)

pytorch_total_params = sum(p.numel() for p in net.net.parameters() if p.requires_grad)
print(pytorch_total_params)

if vol is not None:
    net.eval(vol)
else:
    net.eval(save_period=net.params.save_period)

model_dir = f'{datetime.now().strftime("%Y%m%d_%H%M")}_{net.params.max_steps}steps_{net.params.lr}lr_{net.params.noise_regularizer:.2E}TV_{net.params.output_activation}Act_{net.params.wandb_name}'

model_dir = os.path.join(directory, model_dir)
Path(model_dir).mkdir(parents=True, exist_ok=True)

Best params: {'style_size': 8, 'depth': 4, 'hidden_dim': 256, 'noise_regularizer': 1.497638067198429e-05}  ->  up_factor=16
Epochs: None, Max Steps: 48000, Step Size: 3000
06/16/2026 11:48:39 AM proj_size: 128
06/16/2026 11:48:39 AM frames: 50
06/16/2026 11:48:39 AM dev: cuda:0
06/16/2026 11:48:39 AM seed: 0
06/16/2026 11:48:39 AM opt_over: net
06/16/2026 11:48:39 AM latent_dim: 4
06/16/2026 11:48:39 AM hidden_dim: 256
06/16/2026 11:48:39 AM style_size: 8
06/16/2026 11:48:39 AM depth: 4
06/16/2026 11:48:39 AM Nr: 1
06/16/2026 11:48:39 AM input_nch: 1
06/16/2026 11:48:39 AM output_nch: 1
06/16/2026 11:48:39 AM need_bias: False
06/16/2026 11:48:39 AM up_factor: 16
06/16/2026 11:48:39 AM upsample_mode: nearest
06/16/2026 11:48:39 AM lr: 0.0001
06/16/2026 11:48:39 AM step_size: 3000
06/16/2026 11:48:39 AM gamma: 0.85
06/16/2026 11:48:39 AM batch_size: 4
06/16/2026 11:48:39 AM max_steps: 48000
06/16/2026 11:48:39 AM evaluate: True
06/16/2026 11:48:39 AM eval_vol: False
06/16/2026 11:48:39 A

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\rgirod\_netrc.
wandb: Currently logged in as: robin-girod (robin-girod-university-of-antwerp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


06/16/2026 11:48:43 AM Popen(['git', 'version'], cwd=c:\Users\rgirod\Data\20260611_DIPSTER, stdin=None, shell=False, universal_newlines=False)
06/16/2026 11:48:43 AM Popen(['git', 'version'], cwd=c:\Users\rgirod\Data\20260611_DIPSTER, stdin=None, shell=False, universal_newlines=False)
06/16/2026 11:48:43 AM sys.platform='win32', git_executable='git'


In [17]:
# Run training
net.train(ts, show_summary=True, save_path=model_dir)

# Save results, model weights are saved as '.pkl' files
out_name = net.params.wandb_name        # auto coolname
model_path = os.path.join(model_dir, out_name +'_final.pkl')
torch.save(net.state_dict(), model_path)

Input shape: (4, 1, 1)
MappingNet(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): ReLU()
    (8): Linear(in_features=256, out_features=256, bias=True)
    (9): ReLU()
    (10): Linear(in_features=256, out_features=64, bias=True)
  )
)
Net(
  (net): Sequential(
    (0): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): Upsample(scale_fact

### Inference

Once a network is trained, it gains the capability to output volumes (slice-by-slice) at queried time and angle on the (t, $\theta$) manifold. 

By default, we pass the same times and angles as those of the tiltsereies, but arbitrary values can be queried.

In practice, it is often benefitial to query a single angle (e.g., 0°) and times in-between those of the projection (see the paper). This, way the interpolating nature of the network helps smoothing out the remaining artefacts (misalignments, scan distortions, contrast / brightness changes, etc.) that may not have been sufficiently corrected during preprocessing, and might be learned as wrong physical transformations

In [ ]:
# If needed, reload a previous model, otherwise the last step of training is used
# Training also saves the model with best SSIM and PSNR performance, it is typically best to load one of those
model_path = r"C:\Users\rgirod\Data\20260611_DIPSTER\dipster_bare\data\20260701_NS220_melting\Results\20260701_1202_150000steps_0.0001lr_1.00E-07TV_NoneAffine_reluAct_frosty-totem-1\frosty-totem-1_best_ssim_132000.pkl"
net = dip.Solver.from_state_dict(torch.load(model_path, map_location=torch.device('cuda')))

09/07/2026 02:57:05 PM proj_size: 128
09/07/2026 02:57:05 PM frames: 100
09/07/2026 02:57:05 PM dev: cuda:0
09/07/2026 02:57:05 PM seed: 0
09/07/2026 02:57:05 PM opt_over: net
09/07/2026 02:57:05 PM latent_dim: 4
09/07/2026 02:57:05 PM hidden_dim: 512
09/07/2026 02:57:05 PM style_size: 8
09/07/2026 02:57:05 PM depth: 2
09/07/2026 02:57:05 PM Nr: 1
09/07/2026 02:57:05 PM input_nch: 1
09/07/2026 02:57:05 PM output_nch: 1
09/07/2026 02:57:05 PM need_bias: False
09/07/2026 02:57:05 PM up_factor: 16
09/07/2026 02:57:05 PM upsample_mode: nearest
09/07/2026 02:57:05 PM output_activation: relu
09/07/2026 02:57:05 PM lr: 0.0001
09/07/2026 02:57:05 PM step_size: 9375
09/07/2026 02:57:05 PM gamma: 0.8
09/07/2026 02:57:05 PM batch_size: 4
09/07/2026 02:57:05 PM max_steps: 150000
09/07/2026 02:57:05 PM affine_start_step: None
09/07/2026 02:57:05 PM affine_lr: 5e-06
09/07/2026 02:57:05 PM affine_regularizer: 1
09/07/2026 02:57:05 PM affine_frames_per_step: 1
09/07/2026 02:57:05 PM compile_net: True


In [ ]:
# By default, we export volumes at the same time and angles as the tilt series -> this will be read for the ts sinogram variable
times = ts.times.cpu().numpy()          # in [0,1]
angles = ts.angles.cpu().numpy()        # deg
suffix = ''                             # flag to facilitate naming

# Another option is to use an arbitrary manifold, typically at 0°
# Comment these lines out to keep the training time-angle coordinates
A = 0                                   # angle (°)
angles = np.zeros(ts.angles.shape) + A
suffix = f'-{A}deg-manifold_{times.shape[0]}frames'

# --------------------------------------------------------------------------------------------------------------------
# Save series of .rec files in a folder with the model name
results_dir = model_path.replace('.pkl', f'{suffix}')
Path(results_dir).mkdir(parents=True, exist_ok=True)

full_rec = []
i = 0

for vol, t in net.reconstruct(ts=ts, slice_by_slice=True, times = times, angles = angles):

    # Save vol series
    vol = np.rot90(vol.transpose(2,1,0), 1, axes = (1,2))

    with mrcfile.new(os.path.join(results_dir, f'{i}_data_{times[i]:.3f}t_{angles[i]}deg.rec'), overwrite=True) as mrc:
        mrc.set_data(vol.astype(np.float32))
        mrc.voxel_size = 1280  # Arbitrary value for training, use yours
        mrc.update_header_from_data()
    full_rec.append(vol)
    i+=1

full_rec = np.array(full_rec)
stackview.slice(full_rec, zoom_factor=2, display_min=full_rec.min(), display_max=full_rec.max())

Reconstructing:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
stackview.orthogonal(full_rec[0], zoom_factor=2, display_min=0, display_max = full_rec[0].max())

In [ ]:
# Export synthetic tiltseries for comparison
from dipster_bare import tomo
from dipster_bare.util import np_to_torch
from tqdm.notebook import trange

P = net.params.proj_size
dev = net.params.dev

times = ts.times      
angles = ts.angles
# angles = torch.tensor(np.ones(len(times)) * 45)

depths = torch.arange(P, device=dev)
synth = np.zeros((len(angles), P, P), dtype=np.float32)

for f in trange(len(angles)):
    ang = angles[f].reshape(1).expand(P)
    tim = times[f].reshape(1).expand(P)

    vol = net.reconstruct_slices(ang, depths, tim)             
    synth[f, :, :] = tomo.fp(vol, angles[f]).cpu().squeeze()

io.volwrite(f'{results_dir}_TiltSeries.tif', synth.astype(np.float32))

stackview.slice(synth, zoom_factor=2, display_min=0, display_max=1)

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
io.volwrite(f'{results_dir}_TiltSeries_Raw.tif', np.transpose(ts.data.cpu().numpy().squeeze().astype(np.float32), (2, 0, 1)))

In [ ]:
I = np.argsort(angles.cpu())
stackview.slice(synth[I], zoom_factor=2, display_min=0, display_max=1)